# ScrapSense — Open-Set Waste Classification + Active Learning

Pipeline for classifying waste material images (cardboard, glass, metal, paper,
plastic, trash) with an **open-set recognition** layer: low-confidence
predictions are routed to a **vision-LLM fallback**, and repeated fallback
labels get accumulated toward promoting brand-new classes (**active learning**).

Base dataset: [Garbage Classification](https://www.kaggle.com/datasets/asdasdasdas/garbage-classification)
(6 classes: `cardboard`, `glass`, `metal`, `paper`, `plastic`, `trash`).

**This version reflects settled decisions from experimentation:**
- Moderate regularization (dropout 0.4/0.3, L2, light label smoothing) — best
  balance of closing the train/val gap without giving up accuracy.
- Fine-tuning the EfficientNet backbone is **not** used in the final model —
  tested across multiple regularization configs and consistently reduced val
  accuracy at this dataset size (~2000 train images). Code is kept, commented
  out, for when the dataset grows.
- Class weighting for the underrepresented `trash` class was tested and
  reverted — it improved trash recall but cost more on paper/plastic with no
  net macro-f1 gain.
- Open-set confidence threshold set to **0.35** based on a threshold sweep.

**Sections**
1. Setup & imports
2. Data loading (Kaggle-native or kagglehub, auto-detected)
3. Model — EfficientNetB0 transfer learning (frozen backbone)
4. Training + curves
5. Open-set recognition module
6. Vision LLM fallback (Claude, with disk cache)
7. Active learning accumulator
8. Evaluation (confusion matrix, report, confidence distributions)
9. End-to-end inference demo
10. Save & export for integration


## 1. Setup & imports

In [ ]:
# Optional installs — uncomment if running in a fresh environment (Colab
# already has tensorflow/pandas/matplotlib, but not anthropic/kagglehub)
# !pip install anthropic kagglehub -q

In [ ]:
import os
import json
import shutil
import base64
import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input

from sklearn.metrics import confusion_matrix, classification_report

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))

## 2. Data loading

Points at the `Garbage Classification` dataset's nested folder structure
(`cardboard/`, `glass/`, `metal/`, `paper/`, `plastic/`, `trash/`), and
resolves the actual directory automatically:

1. **`SCRAPSENSE_DATA_DIR` env var**, if set — always wins.
2. **Kaggle Notebook** — walks `/kaggle/input` and finds whichever folder
   actually contains all 6 class subfolders (robust to however the dataset
   got mounted/named, which varies).
3. **Anywhere else (Colab, local)** — downloads via `kagglehub`. Needs Kaggle
   API credentials: get a token from
   [kaggle.com/settings](https://www.kaggle.com/settings) → **API** →
   **Create New Token**, then either upload `kaggle.json` to
   `/content/kaggle.json` (Colab file browser) or set `KAGGLE_USERNAME` /
   `KAGGLE_KEY` as env vars / Colab secrets.

In [ ]:
def ensure_kaggle_credentials():
    """
    Wire up Kaggle API credentials for kagglehub, from whichever source is
    available: an uploaded kaggle.json, or KAGGLE_USERNAME / KAGGLE_KEY env
    vars (e.g. set via Colab's Secrets panel). No-op inside Kaggle Notebooks.
    """
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_json = kaggle_dir / "kaggle.json"

    if kaggle_json.exists() or (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")):
        return

    uploaded = Path("/content/kaggle.json")
    if uploaded.exists():
        kaggle_dir.mkdir(exist_ok=True)
        shutil.copy(uploaded, kaggle_json)
        os.chmod(kaggle_json, 0o600)
        print("Kaggle credentials copied from /content/kaggle.json")
        return

    print(
        "No Kaggle credentials found. If get_dataset_dir() below needs "
        "kagglehub, either upload kaggle.json to /content/kaggle.json and "
        "re-run, or set KAGGLE_USERNAME/KAGGLE_KEY env vars first."
    )


ensure_kaggle_credentials()

In [ ]:
# ============================================================
# CONFIG
# ============================================================
CLASS_NAMES = ["cardboard", "glass", "metal", "paper", "plastic", "trash"]
NUM_CLASSES = len(CLASS_NAMES)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42
VAL_SPLIT = 0.2

# Settled on via threshold sweep during evaluation — see Section 8.
DEFAULT_THRESHOLD = 0.35


def find_dataset_dir(root="/kaggle/input", class_names=CLASS_NAMES):
    """Walk `root` and return the first folder directly containing all class subfolders."""
    if not os.path.isdir(root):
        return None
    for dirpath, dirnames, _ in os.walk(root):
        if all(cls in dirnames for cls in class_names):
            return dirpath
    return None


def get_dataset_dir():
    """Resolve the dataset directory across Kaggle / Colab / local environments."""
    env_override = os.environ.get("SCRAPSENSE_DATA_DIR")
    if env_override and os.path.isdir(env_override):
        return env_override

    kaggle_dir = find_dataset_dir()
    if kaggle_dir:
        return kaggle_dir

    import kagglehub
    path = kagglehub.dataset_download("asdasdasdas/garbage-classification")
    nested = os.path.join(path, "Garbage classification", "Garbage classification")
    return nested if os.path.isdir(nested) else path


DATASET_DIR = get_dataset_dir()
print("Using dataset dir:", DATASET_DIR)
assert DATASET_DIR and os.path.isdir(DATASET_DIR), (
    "Could not resolve a dataset directory. Check Kaggle credentials above, "
    "or set SCRAPSENSE_DATA_DIR directly."
)

In [ ]:
def build_datasets(data_dir, img_size=IMG_SIZE, batch_size=BATCH_SIZE,
                    val_split=VAL_SPLIT, seed=SEED, class_names=CLASS_NAMES):
    """Build train/val tf.data.Dataset objects from a directory of class subfolders."""
    train_ds = tf.keras.utils.image_dataset_from_directory(
        data_dir,
        validation_split=val_split,
        subset="training",
        seed=seed,
        image_size=img_size,
        batch_size=batch_size,
        class_names=class_names,
        label_mode="categorical",
    )
    val_ds = tf.keras.utils.image_dataset_from_directory(
        data_dir,
        validation_split=val_split,
        subset="validation",
        seed=seed,
        image_size=img_size,
        batch_size=batch_size,
        class_names=class_names,
        label_mode="categorical",
    )
    return train_ds, val_ds


train_ds_raw, val_ds_raw = build_datasets(DATASET_DIR)

AUTOTUNE = tf.data.AUTOTUNE

# Final augmentation settings, used throughout experimentation
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.15),
    layers.RandomTranslation(0.1, 0.1),
], name="data_augmentation")


def prep(ds, training=False):
    """Apply EfficientNet preprocessing (+ augmentation for training) and prefetch."""
    ds = ds.map(lambda x, y: (preprocess_input(x), y), num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y),
                     num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)


train_ds = prep(train_ds_raw, training=True)
val_ds = prep(val_ds_raw, training=False)

print("Class names:", CLASS_NAMES)

## 3. Model — EfficientNetB0 transfer learning

Frozen backbone with a regularized head. Dropout (0.4/0.3), a small dense
layer with L2 weight decay, and light label smoothing at compile time — this
combination gave the best balance of closing the train/val gap without
giving up validation accuracy, out of several configs tested.

In [ ]:
def build_model(num_classes=NUM_CLASSES, img_size=IMG_SIZE, base_trainable=False):
    """EfficientNetB0 transfer-learning model with a regularized classification head."""
    inputs = keras.Input(shape=(*img_size, 3))
    base_model = EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_tensor=inputs,
        pooling="avg",
    )
    base_model.trainable = base_trainable

    x = layers.Dropout(0.4)(base_model.output)
    x = layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="predictions")(x)

    model = keras.Model(inputs, outputs)
    return model, base_model


model, base_model = build_model(base_trainable=False)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=["accuracy"],
)
model.summary()

## 4. Training

Frozen-backbone phase only. `ModelCheckpoint` saves the best-val-loss
weights to disk, so the model in memory after this cell may not be the best
one seen — the next cell reloads from the checkpoint explicitly.

In [ ]:
CHECKPOINT_PATH = "scrapsense_best.keras"

callbacks_phase1 = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint(CHECKPOINT_PATH, monitor="val_loss", save_best_only=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2),
]

EPOCHS_PHASE1 = 15

history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE1,
    callbacks=callbacks_phase1,
)

In [ ]:
# Reload the best checkpoint explicitly — don't rely on whatever's left in
# memory after fit(), since EarlyStopping's restore_best_weights and the
# saved checkpoint can diverge if you re-run cells out of order.
model = keras.models.load_model(CHECKPOINT_PATH)
print("Loaded best checkpoint:", CHECKPOINT_PATH)

In [ ]:
def plot_training_curves(history_dict, phase_boundary=None):
    """Plot accuracy and loss curves. Optionally mark where fine-tuning began."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(history_dict["accuracy"], label="train")
    axes[0].plot(history_dict["val_accuracy"], label="val")
    if phase_boundary:
        axes[0].axvline(phase_boundary, linestyle="--", color="gray", label="fine-tune start")
    axes[0].set_title("Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(history_dict["loss"], label="train")
    axes[1].plot(history_dict["val_loss"], label="val")
    if phase_boundary:
        axes[1].axvline(phase_boundary, linestyle="--", color="gray", label="fine-tune start")
    axes[1].set_title("Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    plt.tight_layout()
    plt.show()


plot_training_curves(history_phase1.history)

### Optional: fine-tuning (tested, not used in the final model)

Across every regularization config tested, unfreezing and fine-tuning the
last 30 EfficientNet layers **consistently reduced validation accuracy**
compared to the frozen-backbone model above — likely because ~2000 training
images isn't enough to safely fine-tune that many pretrained layers without
drifting off good ImageNet features.

Kept here for reference (with a checkpoint bug fixed — the original version
let a worse phase-2 checkpoint silently overwrite the better phase-1 one) in
case the dataset grows later. **Every cell below this one uses the
frozen-backbone `model` reloaded above — this section is not run.**

```python
def unfreeze_top_layers(base_model, n_layers=30):
    """Unfreeze the last n_layers of the base model for fine-tuning."""
    base_model.trainable = True
    for layer in base_model.layers[:-n_layers]:
        layer.trainable = False
    return base_model


FINE_TUNE_LAYERS = 30
EPOCHS_PHASE2 = 15

base_model = unfreeze_top_layers(base_model, n_layers=FINE_TUNE_LAYERS)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=["accuracy"],
)

# Prevents phase 2 from overwriting a better phase-1 checkpoint
best_val_loss_phase1 = min(history_phase1.history["val_loss"])
callbacks_phase2 = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint(
        CHECKPOINT_PATH, monitor="val_loss", save_best_only=True,
        initial_value_threshold=best_val_loss_phase1,
    ),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2),
]

history_phase2 = model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS_PHASE2, callbacks=callbacks_phase2
)
model = keras.models.load_model(CHECKPOINT_PATH)  # reload best across both phases
```

## 5. Open-set recognition module

Rejects low-confidence predictions as `"unknown"` instead of forcing them
into one of the known classes. Threshold defaults to `DEFAULT_THRESHOLD`
(0.35), chosen via a threshold sweep on the validation set.

In [ ]:
def predict_with_open_set(softmax_probs, class_names=CLASS_NAMES, threshold=DEFAULT_THRESHOLD):
    """
    Convert a softmax probability array into a (label, confidence) decision.

    If the top class's probability is below `threshold`, the sample is
    considered out-of-distribution and labeled "unknown" instead of being
    forced into one of the known classes.
    """
    top_idx = int(np.argmax(softmax_probs))
    confidence = float(np.max(softmax_probs))

    if confidence < threshold:
        return "unknown", confidence
    return class_names[top_idx], confidence


def predict_image_open_set(model, image_path, class_names=CLASS_NAMES,
                            threshold=DEFAULT_THRESHOLD, img_size=IMG_SIZE):
    """Load a single image from disk, run the model, and apply open-set logic."""
    img = keras.utils.load_img(image_path, target_size=img_size)
    arr = keras.utils.img_to_array(img)
    arr = preprocess_input(arr)
    arr = np.expand_dims(arr, axis=0)

    probs = model.predict(arr, verbose=0)[0]
    label, confidence = predict_with_open_set(probs, class_names, threshold)
    return label, confidence, probs

## 6. Vision LLM fallback

Sends low-confidence images to Claude for classification, with a disk
cache so repeat images never trigger a second paid API call.

Set `ANTHROPIC_API_KEY` as an environment variable — never hardcode it.

In [ ]:
import anthropic

ANTHROPIC_MODEL = "claude-sonnet-4-6"
LLM_CACHE_PATH = "vision_llm_cache.json"


def _load_cache(cache_path=LLM_CACHE_PATH):
    if os.path.exists(cache_path):
        with open(cache_path, "r") as f:
            return json.load(f)
    return {}


def _save_cache(cache, cache_path=LLM_CACHE_PATH):
    with open(cache_path, "w") as f:
        json.dump(cache, f, indent=2)


def _image_to_base64(image_path):
    with open(image_path, "rb") as f:
        return base64.standard_b64encode(f.read()).decode("utf-8")


def _media_type_for(image_path):
    ext = Path(image_path).suffix.lower()
    return {
        ".jpg": "image/jpeg",
        ".jpeg": "image/jpeg",
        ".png": "image/png",
        ".webp": "image/webp",
    }.get(ext, "image/jpeg")


def classify_with_vision_llm(image_path, cache_path=LLM_CACHE_PATH,
                              known_classes=CLASS_NAMES, client=None):
    """
    Classify an "unknown" waste image using a vision LLM as a fallback.

    Caches results to a JSON file keyed by absolute image path, so the
    same image never triggers a second (paid) API call.
    """
    abs_path = str(Path(image_path).resolve())
    cache = _load_cache(cache_path)

    if abs_path in cache:
        return cache[abs_path]

    if client is None:
        client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from env

    prompt = (
        "You are helping classify a waste material for a recycling sorting system. "
        f"Known categories are: {', '.join(known_classes)}. "
        "Look at the image and decide the material category. If it clearly doesn't "
        "fit any known category, propose a short new category name instead. "
        "Respond with ONLY a JSON object of the form "
        '{"category": "<name>", "confidence": <float between 0 and 1>} '
        "and nothing else."
    )

    response = client.messages.create(
        model=ANTHROPIC_MODEL,
        max_tokens=200,
        messages=[{
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "source": {
                        "type": "base64",
                        "media_type": _media_type_for(abs_path),
                        "data": _image_to_base64(abs_path),
                    },
                },
                {"type": "text", "text": prompt},
            ],
        }],
    )

    text = "".join(block.text for block in response.content if block.type == "text").strip()
    try:
        result = json.loads(text)
    except json.JSONDecodeError:
        # Fall back to a safe default if the model didn't return clean JSON
        result = {"category": "unknown", "confidence": 0.0}

    cache[abs_path] = result
    _save_cache(cache, cache_path)
    return result

## 7. Active learning accumulator

Logs every LLM-labeled "unknown" example, and checks when a new label
has enough examples to be promoted into a real training class.

In [ ]:
ACTIVE_LEARNING_LOG_PATH = "active_learning_log.csv"


def log_unknown_example(image_path, llm_label, llm_confidence,
                         log_path=ACTIVE_LEARNING_LOG_PATH):
    """Append one LLM-labeled 'unknown' example to the active learning log (CSV)."""
    row = {
        "image_path": str(image_path),
        "llm_label": llm_label,
        "llm_confidence": llm_confidence,
        "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
    }
    df_row = pd.DataFrame([row])

    if os.path.exists(log_path):
        df_row.to_csv(log_path, mode="a", header=False, index=False)
    else:
        df_row.to_csv(log_path, mode="w", header=True, index=False)

    return row


def load_active_learning_log(log_path=ACTIVE_LEARNING_LOG_PATH):
    if os.path.exists(log_path):
        return pd.read_csv(log_path)
    return pd.DataFrame(columns=["image_path", "llm_label", "llm_confidence", "timestamp"])


def check_new_class_ready(log_df, min_examples=20):
    """
    Check whether any llm_label has accumulated enough examples to be
    promoted to a real training class. Returns a list of ready labels.
    """
    if log_df.empty:
        return []
    counts = log_df["llm_label"].value_counts()
    ready = counts[counts >= min_examples].index.tolist()
    return ready


def promote_class(label, log_df, dataset_dir=DATASET_DIR, class_names=CLASS_NAMES):
    """
    Copy all logged images for `label` into a new class folder under
    dataset_dir, and print instructions for re-running training.

    This mutates the on-disk dataset structure directly — double-check
    log_df before calling this for real.
    """
    new_class_dir = os.path.join(dataset_dir, label)
    os.makedirs(new_class_dir, exist_ok=True)

    matching = log_df[log_df["llm_label"] == label]
    moved = 0
    for _, row in matching.iterrows():
        src = row["image_path"]
        if os.path.exists(src):
            dst = os.path.join(new_class_dir, os.path.basename(src))
            shutil.copy2(src, dst)
            moved += 1

    updated_class_names = class_names + [label] if label not in class_names else class_names

    print(f"Promoted '{label}': copied {moved} images to {new_class_dir}")
    print("Next steps:")
    print(f"  1. Update CLASS_NAMES to: {updated_class_names}")
    print("  2. Re-run build_datasets() to rebuild train/val splits")
    print("  3. Re-run build_model() with the new NUM_CLASSES and retrain")

    return updated_class_names

## 8. Evaluation

Uses `DEFAULT_THRESHOLD` (0.35). If you change the model or retrain, re-run
a threshold sweep (varying `threshold` in `evaluate_on_val_set`) rather than
assuming 0.35 still holds — different regularization or class balancing
shifts the confidence distribution.

In [ ]:
def evaluate_on_val_set(model, val_ds_raw, class_names=CLASS_NAMES, threshold=DEFAULT_THRESHOLD):
    """
    Run the full open-set pipeline over the validation set and return
    true labels, predicted labels (including 'unknown'), and confidences.
    """
    y_true, y_pred, confidences = [], [], []

    for images, labels in val_ds_raw:
        processed = preprocess_input(images.numpy().copy())
        probs_batch = model.predict(processed, verbose=0)

        true_idx = np.argmax(labels.numpy(), axis=1)
        for probs, t_idx in zip(probs_batch, true_idx):
            label, conf = predict_with_open_set(probs, class_names, threshold)
            y_true.append(class_names[t_idx])
            y_pred.append(label)
            confidences.append(conf)

    return y_true, y_pred, confidences


y_true, y_pred, confidences = evaluate_on_val_set(model, val_ds_raw, threshold=DEFAULT_THRESHOLD)

labels_for_cm = CLASS_NAMES + (["unknown"] if "unknown" in y_pred else [])
cm = confusion_matrix(y_true, y_pred, labels=labels_for_cm)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels_for_cm, yticklabels=labels_for_cm)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix (open-set aware)")
plt.tight_layout()
plt.show()

print(classification_report(y_true, y_pred, labels=labels_for_cm, zero_division=0))

In [ ]:
def plot_confidence_distribution(y_pred, confidences, threshold=DEFAULT_THRESHOLD):
    """Plot confidence distributions separately for 'known' vs 'unknown' predictions."""
    confidences = np.array(confidences)
    y_pred = np.array(y_pred)

    known_conf = confidences[y_pred != "unknown"]
    unknown_conf = confidences[y_pred == "unknown"]

    plt.figure(figsize=(8, 5))
    plt.hist(known_conf, bins=20, alpha=0.6, label="known predictions")
    plt.hist(unknown_conf, bins=20, alpha=0.6, label="unknown predictions")
    plt.axvline(threshold, linestyle="--", color="gray", label=f"threshold ({threshold})")
    plt.xlabel("Confidence (max softmax)")
    plt.ylabel("Count")
    plt.title("Confidence Distribution: Known vs Unknown")
    plt.legend()
    plt.tight_layout()
    plt.show()


plot_confidence_distribution(y_pred, confidences)

**Model selection summary** (from experimentation, kept here for the writeup):

| Config tested | Train/val gap | Val ceiling accuracy | Kept? |
|---|---|---|---|
| No regularization | Large (~0.07) | ~0.88 | No — overfit |
| Heavy regularization (dropout 0.5/0.4, label smoothing 0.1) | ~0 | ~0.865 | No — underfit, gave back too much accuracy |
| **Moderate regularization (this notebook)** | **~0.02–0.03** | **~0.87** | **Yes** |
| + class weighting for `trash` | same | ~0.87 (same headline, worse macro-f1) | No — helped trash recall, hurt paper/plastic, net-neutral |
| + fine-tuning (30 layers unfrozen) | — | consistently *lower* than frozen backbone | No — not enough data for this dataset size |


## 9. End-to-end inference demo

Predict → check confidence → fall back to the vision LLM if needed →
log if unknown → report whether any new class is ready for promotion.

In [ ]:
def classify_waste_image(image_path, model, class_names=CLASS_NAMES,
                          threshold=DEFAULT_THRESHOLD, min_examples_for_promotion=20):
    """
    Full ScrapSense inference pipeline for a single image:
      1. Predict with the CNN.
      2. Apply the open-set threshold.
      3. If unknown, fall back to the vision LLM and log the result.
      4. Check whether the resulting label is ready to be promoted to a real class.

    Returns a dict describing the final decision.
    """
    label, confidence, probs = predict_image_open_set(
        model, image_path, class_names, threshold
    )

    decision = {
        "image_path": image_path,
        "cnn_label": label,
        "cnn_confidence": confidence,
        "source": "cnn",
    }

    if label == "unknown":
        llm_result = classify_with_vision_llm(image_path, known_classes=class_names)
        log_unknown_example(image_path, llm_result["category"], llm_result["confidence"])

        decision["final_label"] = llm_result["category"]
        decision["final_confidence"] = llm_result["confidence"]
        decision["source"] = "vision_llm"

        log_df = load_active_learning_log()
        ready_labels = check_new_class_ready(log_df, min_examples=min_examples_for_promotion)
        decision["classes_ready_for_promotion"] = ready_labels
    else:
        decision["final_label"] = label
        decision["final_confidence"] = confidence
        decision["classes_ready_for_promotion"] = []

    return decision


# --- Demo run ---
# Point this at any single image to see the full pipeline in action.
DEMO_IMAGE_PATH = "path/to/a/test_image.jpg"

if os.path.exists(DEMO_IMAGE_PATH):
    result = classify_waste_image(DEMO_IMAGE_PATH, model, threshold=DEFAULT_THRESHOLD)
    print(json.dumps(result, indent=2))
else:
    print(f"Set DEMO_IMAGE_PATH to a real image file to run the demo "
          f"(currently: {DEMO_IMAGE_PATH})")

## 10. Save & export for integration

Bundles the trained model with a small config file (class names, image
size, chosen threshold) so the rest of ScrapSense's pipeline — e.g. an
agent/subagent that calls this model — can load both without hardcoding
these values a second time somewhere else.

In [ ]:
EXPORT_DIR = "scrapsense_export"
os.makedirs(EXPORT_DIR, exist_ok=True)

final_model_path = os.path.join(EXPORT_DIR, "scrapsense_model.keras")
model.save(final_model_path)

config = {
    "class_names": CLASS_NAMES,
    "img_size": list(IMG_SIZE),
    "open_set_threshold": DEFAULT_THRESHOLD,
    "model_path": final_model_path,
    "notes": (
        "Frozen-backbone EfficientNetB0 head only. Fine-tuning and class "
        "weighting were both tested and did not improve on this config at "
        "this dataset size — see the model selection summary in Section 8."
    ),
}
config_path = os.path.join(EXPORT_DIR, "config.json")
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print(f"Exported model + config to ./{EXPORT_DIR}/")
print(json.dumps(config, indent=2))

**To load this elsewhere in the ScrapSense codebase:**

```python
import json
from tensorflow import keras

with open("scrapsense_export/config.json") as f:
    config = json.load(f)

model = keras.models.load_model(config["model_path"])
CLASS_NAMES = config["class_names"]
THRESHOLD = config["open_set_threshold"]
```

Then reuse `predict_with_open_set`, `classify_with_vision_llm`, and
`classify_waste_image` from this notebook as-is (or move them into a shared
module) — none of them depend on anything Kaggle/Colab-specific.